# Day 32 — Time Series Analysis

Topics:
- `pd.to_datetime()` — parsing dates
- Setting datetime as index
- `.dt` accessor — extracting date parts
- Resampling — grouping by time period
- Rolling windows — moving averages
- Shifting — lag/lead comparisons

In [ ]:
import pandas as pd
import numpy as np

## 1. Parsing Dates — `pd.to_datetime()`

Pandas stores dates as `datetime64` — a special dtype that unlocks time-based operations.
Without this, dates are just strings and you can't do any time math.

In [ ]:
# String dates — pandas doesn't know these are dates yet
dates = ["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"]

s = pd.Series(dates)
print("Before:", s.dtype)  # object (string)

s = pd.to_datetime(s)
print("After: ", s.dtype)  # datetime64[ns]

In [ ]:
# pandas can parse many formats automatically
pd.to_datetime(["01/15/2024", "Jan 15 2024", "2024-01-15", "15-01-2024"])

In [ ]:
# if the format is unusual, specify it explicitly
pd.to_datetime(["15/01/2024"], format="%d/%m/%Y")

## 2. Building a Time Series Dataset

In [ ]:
# pd.date_range — generate a sequence of dates
dates = pd.date_range(start="2024-01-01", periods=365, freq="D")

np.random.seed(42)
df = pd.DataFrame({
    "date":    dates,
    "revenue": np.random.randint(1000, 5000, size=365) + np.arange(365) * 5,  # upward trend
    "users":   np.random.randint(100, 500, size=365),
    "region":  np.random.choice(["North", "South", "East"], size=365)
})

df.head()

## 3. Setting Datetime as Index

Setting the date column as the index unlocks time-based slicing and resampling.
This is the standard way to work with time series in pandas.

In [ ]:
df = df.set_index("date")
df.head()

In [ ]:
# time-based slicing — only works when date is the index
df["2024-03"]                        # all of March


In [ ]:
df["2024-01-01" : "2024-01-31"]      # January only

In [ ]:
df["2024-Q2"]                        # second quarter

## 4. The `.dt` Accessor — Extracting Date Parts

`.dt` gives you access to date components (month, day, weekday, etc.) from a datetime column.
Think of it like `.str` but for dates.

In [ ]:
# reset index to use .dt on a column (not index)
df_col = df.reset_index()

df_col["month"]     = df_col["date"].dt.month        # 1-12
df_col["month_name"]= df_col["date"].dt.month_name() # January, February...
df_col["day"]       = df_col["date"].dt.day           # 1-31
df_col["weekday"]   = df_col["date"].dt.day_name()    # Monday, Tuesday...
df_col["week"]      = df_col["date"].dt.isocalendar().week  # week number
df_col["quarter"]   = df_col["date"].dt.quarter       # 1-4
df_col["is_weekend"]= df_col["date"].dt.dayofweek >= 5  # True/False

df_col[["date", "month_name", "weekday", "quarter", "is_weekend"]].head(10)

In [ ]:
# practical use: which weekday has highest average revenue?
df_col.groupby("weekday")["revenue"].mean().sort_values(ascending=False)

## 5. Resampling — Grouping by Time Period

`resample()` is like `groupby()` but for time periods.
It collapses rows into time buckets (weekly, monthly, quarterly, etc.).

Common freq aliases:
- `"D"` — daily
- `"W"` — weekly
- `"ME"` — month end
- `"QE"` — quarter end
- `"YE"` — year end

In [ ]:
# monthly total revenue
df["revenue"].resample("ME").sum()

In [ ]:
# weekly average users
df["users"].resample("W").mean()

In [ ]:
# multiple aggregations at once
df["revenue"].resample("ME").agg(["sum", "mean", "max", "min"])

In [ ]:
# quarterly revenue
df["revenue"].resample("QE").sum()

## 6. Rolling Windows — Moving Averages

`rolling(n)` creates a sliding window of n rows.
Used to smooth out noise and reveal trends.

Real-world use: stock prices, sales trends, server metrics.

In [ ]:
# 7-day rolling average revenue
df["revenue_7d"]  = df["revenue"].rolling(7).mean()

# 30-day rolling average
df["revenue_30d"] = df["revenue"].rolling(30).mean()

df[["revenue", "revenue_7d", "revenue_30d"]].head(35)

In [ ]:
# first n-1 rows are NaN because the window isn't full yet
# rolling(7) needs 7 rows before it can compute — rows 0-5 are NaN
df["revenue_7d"].isna().sum()  # 6 NaN values

In [ ]:
# rolling works with other aggregations too
df["revenue"].rolling(7).max()   # rolling max
df["revenue"].rolling(7).std()   # rolling volatility

## 7. Shifting — Lag and Lead Comparisons

`shift(n)` moves values forward (positive) or backward (negative) by n periods.
Used to compare current value against a past or future value.

In [ ]:
# day-over-day revenue change
df["prev_day_revenue"] = df["revenue"].shift(1)       # yesterday's revenue
df["daily_change"]     = df["revenue"] - df["revenue"].shift(1)
df["pct_change"]       = df["revenue"].pct_change() * 100  # % change

df[["revenue", "prev_day_revenue", "daily_change", "pct_change"]].head(10)

In [ ]:
# week-over-week comparison (shift by 7)
df["wow_change"] = df["revenue"] - df["revenue"].shift(7)
df[["revenue", "wow_change"]].head(15)

In [ ]:
# shift(-1) = look ahead (lead) — tomorrow's value on today's row
df["next_day_revenue"] = df["revenue"].shift(-1)
df[["revenue", "next_day_revenue"]].tail(5)  # last row will be NaN

## 8. Putting It Together — Mini Analysis

In [ ]:
# Monthly summary: total revenue, avg daily users, best single day
monthly = df.resample("ME").agg(
    total_revenue = ("revenue", "sum"),
    avg_users     = ("users",   "mean"),
    best_day      = ("revenue", "max")
).reset_index()

monthly["month"] = monthly["date"].dt.month_name()
monthly[["month", "total_revenue", "avg_users", "best_day"]]

In [ ]:
# which month had highest revenue?
monthly.loc[monthly["total_revenue"].idxmax(), "month"]

In [ ]:
# month-over-month revenue growth %
monthly["mom_growth"] = monthly["total_revenue"].pct_change() * 100
monthly[["month", "total_revenue", "mom_growth"]]

## Practice Exercises

1. Find the week with the highest total revenue using `resample()`
2. Add a column `revenue_14d` — 14-day rolling average
3. Find which quarter had the most users (sum)
4. Add a column `is_above_30d_avg` — True if that day's revenue is above the 30-day rolling average
5. Find the day with the biggest single-day revenue drop (most negative `daily_change`)

In [ ]:
# your solutions here